# Ad conversion on four channels: Marascuilo procedure

> **Before you run:** choose **Copy to Drive** at the top of this page (or **File → Save a copy in Drive**). Your copy opens in a new tab. In your copy, choose **Runtime → Run all**. Your copy is yours, so Colab runs it without a warning, and it keeps your changes.

**Comparing Multiple Proportions: The Marascuilo Procedure** · a worked example from the StatNex teaching guide

**The story.** A brand ran the same ad on four channels and measured the share of viewers who clicked through and bought: social media, a short-video app, a video site and email.

**The question.** Which channels convert better than others?

If you run this page without a copy, Colab first warns that the notebook was not written by Google. It reads none of your data, so **Run anyway** is safe, but your changes are not kept.

The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `05_anova/proportion_multiple_comparison.csv` (2 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Channel,Social,Short video,Video site,Email
Sample size,400.0,350.0,300.0,450.0
Conversion rate,0.085,0.12,0.1,0.06

## Libraries

In [ ]:
import itertools

import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)

## The data

In [ ]:
# One column per group and two rows of summary numbers; the individual answers never
# enter this method.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
groups = ["Social", "Short video", "Video site", "Email"]
first = data[groups].iloc[0].to_numpy(float)
second = data[groups].iloc[1].to_numpy(float)

# Row 1 is the sample size and row 2 the proportion. A proportion typed as a
# percentage (17 rather than 0.17) is divided by 100 first.
sizes, rates = first, second
if (rates > 1).any():
    rates = rates / 100
counts = rates * sizes

## Descriptive statistics

In [ ]:
descriptive = pd.DataFrame(
    [sizes, rates, counts, sizes - counts],
    index=["n", "proportion", "in the category", "not in the category"], columns=groups,
)
print(descriptive)

## Test of independence

In [ ]:
# Whether the proportions differ at all: the chi-square test on the 2 x k table of
# counts, without Yates' continuity correction.
table = np.vstack([counts, sizes - counts])
chi2, p_value, df, expected = stats.chi2_contingency(table, correction=False)
print(f"\nchi-square = {chi2:.4f}, df = {df}, P = {p_value:.4g}")

## Multiple comparison test (Marascuilo's procedure)

In [ ]:
# Each pair's squared difference over its variance, referred to the chi-square
# distribution with k - 1 degrees of freedom -- the whole table's, not 1. Paying for
# all k - 1 degrees of freedom on every pair is what keeps the familywise error at 5%.
k = len(groups)
rows = []
for i, j in itertools.combinations(range(k), 2):
    difference = rates[i] - rates[j]
    variance = rates[i] * (1 - rates[i]) / sizes[i] + rates[j] * (1 - rates[j]) / sizes[j]
    statistic = difference**2 / variance if variance > 0 else np.nan
    rows.append([groups[i], sizes[i], rates[i], groups[j], sizes[j], rates[j],
                 abs(difference), statistic, k - 1, stats.chi2.sf(statistic, k - 1)])
pairs = pd.DataFrame(rows, columns=["group 1", "n 1", "proportion 1", "group 2", "n 2",
                                    "proportion 2", "difference", "chi-square", "df", "P"])
print()
print(pairs.to_string(index=False))

## What the result says

The conversion rates are 12% for short video, 10% for the video site, 8.5% for social and 6% for email. The test of independence asks whether the four rates could all be the same. Its chi-square is 9.37 with P = 0.025. The P value is the chance of a result at least this extreme if the rates were equal, so some channels differ.

The Marascuilo procedure then compares every pair of proportions and keeps the risk of a false alarm under control. Only one pair is significant: short video against email, a difference of 0.06 with P = 0.038. All other pairs have P = 0.288 or more.

Short video converts better than email. The other gaps are not shown, and the best channel also depends on cost.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.